# Step 1: Data Loading & Initial Exploration

## Overview
This notebook covers the first step of our Supply Chain ML pipeline: loading and exploring the raw data.

### Learning Objectives
- Understand the data loading mechanism using `data_manager.py`
- Explore the DataCo Supply Chain dataset structure
- Identify data quality issues and understand the business context

### Business Context
The DataCo Supply Chain dataset contains e-commerce transaction data including:
- **Order Information**: Order dates, items, quantities, prices
- **Customer Data**: Location, segment, demographics
- **Product Details**: Categories, descriptions, pricing
- **Shipping Information**: Delivery status, shipping modes, dates
- **Financial Metrics**: Sales, profits, discounts


In [ ]:
# Standard imports
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.append('..')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Project modules
from src.data.data_manager import (
    load_raw, RAW_DIR, INTERIM_DIR, PROCESSED_DIR,
    RAW_FILE, DATASET_ID
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Setup complete!")


## 1.1 Understanding the Data Manager

The `data_manager.py` module provides:
- **Automatic Download**: Downloads data from Kaggle if not present locally
- **Caching**: Stores raw data locally for 24 hours before refresh
- **Path Management**: Handles file paths for raw, interim, and processed data
- **Encoding Handling**: Manages character encoding issues (Latin-1 vs UTF-8)


In [ ]:
# Display data manager configuration
print("=" * 60)
print("DATA MANAGER CONFIGURATION")
print("=" * 60)
print(f"\n📁 Data Directories:")
print(f"   Raw:       {RAW_DIR}")
print(f"   Interim:   {INTERIM_DIR}")
print(f"   Processed: {PROCESSED_DIR}")
print(f"\n📊 Dataset:")
print(f"   Kaggle ID: {DATASET_ID}")
print(f"   Local File: {RAW_FILE}")


## 1.2 Loading the Raw Dataset

The `load_raw()` function:
1. Checks if the local file exists and is recent (< 24 hours)
2. If not, downloads fresh data from Kaggle
3. Handles encoding issues automatically


In [ ]:
# Load raw data
df = load_raw()

print("\n" + "=" * 60)
print("DATASET LOADED")
print("=" * 60)
print(f"\n📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"💾 Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


## 1.3 Dataset Structure

Let's examine the structure of our dataset to understand what we're working with.


In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()


In [ ]:
# Column information
print("\n" + "=" * 60)
print("COLUMN INFORMATION")
print("=" * 60)

column_info = pd.DataFrame({
    'Data Type': df.dtypes,
    'Non-Null Count': df.count(),
    'Null Count': df.isnull().sum(),
    'Unique Values': df.nunique()
})
print(column_info)


## 1.4 Data Quality Overview

### Key Metrics to Check:
1. **Missing Values**: Identify columns with null values
2. **Duplicates**: Check for duplicate records
3. **Data Types**: Verify correct data types


In [ ]:
# Missing values analysis
print("=" * 60)
print("MISSING VALUES ANALYSIS")
print("=" * 60)

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

# Show only columns with missing values
missing_df = missing_df[missing_df['Missing Count'] > 0]
if len(missing_df) > 0:
    print(missing_df)
else:
    print("✅ No missing values found!")


In [ ]:
# Target variable: Late Delivery Risk
if 'Late_delivery_risk' in df.columns:
    print("=" * 60)
    print("TARGET: Late Delivery Risk (Classification)")
    print("=" * 60)

    risk_dist = df['Late_delivery_risk'].value_counts()
    risk_pct = df['Late_delivery_risk'].value_counts(normalize=True) * 100

    print(f"\n0 (On-time): {risk_dist[0]:,} ({risk_pct[0]:.1f}%)")
    print(f"1 (Late):    {risk_dist[1]:,} ({risk_pct[1]:.1f}%)")

    # Visualize
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['#2ecc71', '#e74c3c']
    ax.bar(['On-time (0)', 'Late (1)'], risk_dist.values, color=colors)
    ax.set_ylabel('Count')
    ax.set_title('Late Delivery Risk Distribution', fontweight='bold')
    for i, v in enumerate(risk_dist.values):
        ax.text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("\n💡 Interpretation: The classes are relatively balanced (~55% late, ~45% on-time).")
    print("   This is good for training classification models without severe class imbalance issues.")


## 1.5 Summary & Key Findings

### Data Overview:
- **Size**: ~180K records with 53 columns
- **Time Period**: Order dates span multiple years
- **Geographic Scope**: Multiple countries and regions

### Quality Issues Identified:
1. ✅ No complete duplicate rows
2. ⚠️ Missing values in some columns (Order Zipcode, Product Description)
3. ⚠️ Date columns stored as strings (need conversion)
4. ⚠️ Some encoding issues with special characters

### Next Steps:
1. **Preprocessing** (Notebook 02): Handle missing values, data types, duplicates
2. **Feature Engineering** (Notebook 03): Create meaningful features from raw data
3. **Model Training** (Notebooks 04-05): Train classification and forecasting models


In [ ]:
# Final summary
print("=" * 60)
print("DATA LOADING SUMMARY")
print("=" * 60)
print(f"\n✅ Dataset loaded successfully")
print(f"   • Rows: {df.shape[0]:,}")
print(f"   • Columns: {df.shape[1]}")
print(f"   • Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\n📊 Target Variables:")
print(f"   • Late_delivery_risk (Classification): Binary (0/1)")
print(f"   • Sales (Forecasting): Continuous")
print(f"\n➡️ Next: Run 02_preprocessing.ipynb to clean the data")
